# 2.1 Sistemas de Patrones
## Explotación de la Información

**Duración estimada:** 5 horas  
**Dataset:** `hotel_bookings_clean.csv`

### Pregunta central

> ¿Qué patrones, perfiles y segmentos podemos descubrir dentro de un dataset?

En esta sesión no construiremos modelos predictivos.

El objetivo es aprender a **extraer estructura útil de un conjunto de datos**, comenzando con herramientas muy básicas y terminando con **K-Means** como técnica de segmentación automática.

## Ruta de la sesión

### Bloque 1 — Patrones básicos
- Frecuencias absolutas y relativas
- Perfiles y comparación de grupos
- Tablas cruzadas

### Bloque 2 — Construcción de segmentos
- Creación de categorías
- Segmentación por reglas
- Visualización de patrones
- Comportamientos atípicos

### Bloque 3 — Segmentación automática
- Similitud
- Distancia euclidiana
- Escalas
- Estandarización
- K-Means
- Método del codo
- Interpretación de clusters

### Cierre
- Actividad integradora

> **Observar → comparar → segmentar → visualizar → medir similitud → agrupar → interpretar**

## Distribución sugerida de las 5 horas

| Bloque | Tiempo aproximado |
|---|---:|
| Patrones básicos | 60 min |
| Segmentación manual y visualización | 75 min |
| Atípicos, similitud y escalas | 45 min |
| K-Means | 60 min |
| Actividad integradora | 60 min |

# 0. Preparación del entorno

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Cargar el dataset

El notebook supone que el archivo `hotel_bookings_clean.csv` ya está disponible en Colab.

In [ ]:
df = pd.read_csv("hotel_bookings_clean.csv")

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

df.head()

## Crear variables auxiliares

In [ ]:
df["total_nights"] = (
    df["stays_in_week_nights"] +
    df["stays_in_weekend_nights"]
)

df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

df[["total_nights", "total_guests"]].head()

# 1. ¿Qué es un patrón?

Un **patrón** es una regularidad observable dentro de los datos.

Puede manifestarse como:

- una categoría dominante;
- una diferencia entre grupos;
- una combinación frecuente;
- una concentración de valores;
- una tendencia;
- un comportamiento atípico;
- un segmento de observaciones similares.

En esta clase no buscamos “el algoritmo correcto”.

Buscamos responder:

> **¿Qué información útil está escondida dentro del dataset?**

## 1.1 Patrón, tendencia, segmento y anomalía

### Patrón
Regularidad general.

### Tendencia
Comportamiento que cambia sistemáticamente en una dirección.

### Segmento
Grupo de observaciones con características similares.

### Anomalía
Observación poco habitual respecto al resto.

Un mismo dataset puede contener los cuatro.

# 2. Frecuencias absolutas y relativas

La forma más sencilla de encontrar un patrón es contar.

### Frecuencia absoluta
Número de veces que aparece una categoría.

### Frecuencia relativa
Proporción que representa respecto al total.

\[
f_r = \frac{f_i}{N}
\]

In [ ]:
segment_freq = df["market_segment"].value_counts()

segment_table = pd.DataFrame({
    "frecuencia": segment_freq,
    "proporcion": segment_freq / segment_freq.sum(),
    "porcentaje": segment_freq / segment_freq.sum() * 100
})

segment_table.round(3)

In [ ]:
segment_table["porcentaje"].sort_values().plot(
    kind="barh",
    figsize=(9,5)
)

plt.xlabel("Porcentaje de reservas")
plt.ylabel("Segmento de mercado")
plt.title("Distribución por segmento de mercado")
plt.show()

### Preguntas de interpretación

- ¿Cuál es el segmento más frecuente?
- ¿Cuál es el menos frecuente?
- ¿Qué diferencia existe entre decir “hay 20,000 reservas” y “representan 24 % del total”?
- ¿Una categoría muy frecuente necesariamente tiene mejor desempeño?

## Mini ejercicio 1

Repite el análisis para:

- `hotel`
- `customer_type`
- `deposit_type`

Para cada variable incluye frecuencia, porcentaje y una gráfica.

In [ ]:
# Tu código aquí

# 3. Perfiles y comparación de grupos

Contar grupos nos dice cuánto aparece cada uno.

Pero para **explotar información** necesitamos conocer cómo se comportan.

Podemos construir perfiles mediante estadísticas agregadas.

In [ ]:
profile_market = (
    df.groupby("market_segment")
      .agg(
          reservas=("market_segment", "size"),
          adr_promedio=("adr", "mean"),
          anticipacion_promedio=("lead_time", "mean"),
          noches_promedio=("total_nights", "mean"),
          huespedes_promedio=("total_guests", "mean"),
          cancelacion_pct=("is_canceled", lambda s: s.mean() * 100)
      )
      .sort_values("reservas", ascending=False)
      .round(2)
)

profile_market

In [ ]:
profile_market["cancelacion_pct"].sort_values().plot(
    kind="barh",
    figsize=(9,5)
)

plt.xlabel("Cancelación (%)")
plt.title("Cancelación por segmento de mercado")
plt.show()

In [ ]:
profile_market[[
    "adr_promedio",
    "anticipacion_promedio",
    "noches_promedio"
]].plot(
    kind="bar",
    figsize=(11,6)
)

plt.title("Perfil de los segmentos")
plt.ylabel("Valor promedio")
plt.xticks(rotation=45)
plt.show()

### Lectura de un perfil

No basta con decir:

> “Online TA tiene muchas reservas”.

Podemos enriquecerlo:

> “Online TA concentra muchas reservas, tiene determinada anticipación promedio, cierta tarifa promedio y una tasa de cancelación distinta a otros segmentos”.

Eso ya es **información explotada**.

## Mini ejercicio 2

Construye un perfil por `customer_type`.

Incluye:

- número de reservas;
- ADR promedio;
- noches promedio;
- anticipación promedio;
- cancelación promedio.

Después escribe **dos diferencias relevantes** entre grupos.

In [ ]:
# Tu código aquí

# 4. Tablas cruzadas: dos categorías al mismo tiempo

Una frecuencia analiza una variable categórica.

Una **tabla cruzada** permite observar dos variables categóricas simultáneamente.

In [ ]:
hotel_cancel = pd.crosstab(
    df["hotel"],
    df["is_canceled"]
)

hotel_cancel

## 4.1 Tabla cruzada con proporciones

In [ ]:
hotel_cancel_pct = pd.crosstab(
    df["hotel"],
    df["is_canceled"],
    normalize="index"
).mul(100)

hotel_cancel_pct.columns = ["No cancelada", "Cancelada"]

hotel_cancel_pct.round(2)

In [ ]:
hotel_cancel_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(8,5)
)

plt.ylabel("Porcentaje")
plt.title("Composición de cancelaciones por tipo de hotel")
plt.xticks(rotation=0)
plt.show()

### Pregunta

> ¿La categoría con más cancelaciones absolutas es necesariamente la que tiene mayor proporción de cancelación?

Esta diferencia entre **conteo** y **proporción** es fundamental.

## Mini ejercicio 3

Construye y visualiza dos tablas cruzadas:

1. `market_segment × is_canceled`
2. `customer_type × deposit_type`

Escribe una conclusión para cada una.

In [ ]:
# Tu código aquí

# 5. Crear categorías a partir de variables numéricas

Muchas variables son continuas o numéricas.

Podemos transformarlas en categorías para facilitar la interpretación.

Por ejemplo, `lead_time` puede convertirse en niveles de anticipación.

In [ ]:
df["anticipacion"] = pd.cut(
    df["lead_time"],
    bins=[-1, 7, 30, 90, 180, np.inf],
    labels=[
        "0–7 días",
        "8–30 días",
        "31–90 días",
        "91–180 días",
        ">180 días"
    ]
)

df[["lead_time", "anticipacion"]].head(10)

In [ ]:
df["anticipacion"].value_counts(sort=False)

## 5.1 Crear categorías de duración

In [ ]:
df["duracion_estancia"] = pd.cut(
    df["total_nights"],
    bins=[-1, 1, 3, 7, np.inf],
    labels=[
        "1 noche",
        "2–3 noches",
        "4–7 noches",
        ">7 noches"
    ]
)

df[["total_nights", "duracion_estancia"]].head(10)

## 5.2 `pd.cut()` vs `pd.qcut()`

- `cut()` usa límites que nosotros definimos.
- `qcut()` intenta crear grupos con cantidades similares de observaciones.

`cut()` suele ser más fácil de justificar cuando existen reglas de negocio.

In [ ]:
df["adr_quartile"] = pd.qcut(
    df["adr"],
    q=4,
    labels=["Q1 bajo", "Q2", "Q3", "Q4 alto"],
    duplicates="drop"
)

df["adr_quartile"].value_counts(sort=False)

# 6. Segmentación por reglas

Una segmentación por reglas usa criterios definidos por nosotros.

Ejemplo:

> reservas con alta anticipación y estancia larga.

Este enfoque es fácil de interpretar porque conocemos exactamente por qué una observación pertenece a un segmento.

In [ ]:
df["segmento_regla"] = "Otro"

df.loc[
    (df["lead_time"] <= 7) &
    (df["total_nights"] <= 3),
    "segmento_regla"
] = "Última hora + estancia corta"

df.loc[
    (df["lead_time"] > 90) &
    (df["total_nights"] <= 3),
    "segmento_regla"
] = "Anticipada + estancia corta"

df.loc[
    df["total_nights"] >= 7,
    "segmento_regla"
] = "Estancia larga"

df["segmento_regla"].value_counts()

In [ ]:
rule_profile = (
    df.groupby("segmento_regla")
      .agg(
          reservas=("segmento_regla", "size"),
          adr_promedio=("adr", "mean"),
          lead_time_promedio=("lead_time", "mean"),
          noches_promedio=("total_nights", "mean"),
          cancelacion_pct=("is_canceled", lambda s: s.mean() * 100)
      )
      .round(2)
)

rule_profile

## 6.1 Combinar dos categorías

In [ ]:
manual_patterns = (
    df.groupby(
        ["anticipacion", "duracion_estancia"],
        observed=True
    )
    .agg(
        reservas=("is_canceled", "size"),
        cancelacion_pct=("is_canceled", lambda s: s.mean() * 100),
        adr_promedio=("adr", "mean")
    )
    .reset_index()
)

manual_patterns.head()

In [ ]:
heat_cancel = manual_patterns.pivot(
    index="anticipacion",
    columns="duracion_estancia",
    values="cancelacion_pct"
)

plt.figure(figsize=(10,5))
sns.heatmap(
    heat_cancel,
    annot=True,
    fmt=".1f"
)

plt.title("Cancelación (%) por anticipación y duración")
plt.xlabel("Duración")
plt.ylabel("Anticipación")
plt.show()

### Pregunta

> ¿Qué patrones aparecen cuando combinamos variables que no eran tan evidentes al analizarlas por separado?

## Mini ejercicio 4

Crea tu propia segmentación por reglas usando al menos dos variables.

Ejemplos:

- número de huéspedes;
- solicitudes especiales;
- tipo de hotel;
- cliente recurrente;
- depósito;
- segmento de mercado.

Después construye un perfil de tus segmentos.

In [ ]:
# Tu código aquí

# 7. Visualización para encontrar patrones

No todas las gráficas responden la misma pregunta.

### Barras
Comparan categorías.

### Histograma
Muestra concentración y forma de una variable.

### Boxplot
Compara distribuciones entre grupos.

### Scatter plot
Permite observar relaciones, concentraciones y posibles grupos.

### Heatmap
Permite comparar muchas combinaciones de forma compacta.

## 7.1 Histograma

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(
    data=df,
    x="lead_time",
    bins=50
)

plt.xlim(0,500)
plt.title("Distribución de anticipación")
plt.show()

## 7.2 Boxplot

In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(
    data=df,
    x="hotel",
    y="adr"
)

plt.title("Distribución de ADR por hotel")
plt.show()

## 7.3 Scatter plot

In [ ]:
sample_scatter = df.sample(
    min(5000, len(df)),
    random_state=42
)

plt.figure(figsize=(9,6))
sns.scatterplot(
    data=sample_scatter,
    x="lead_time",
    y="adr",
    hue="hotel",
    alpha=0.35
)

plt.title("Lead time vs ADR")
plt.show()

### Mini ejercicio 5

Para cada gráfica anterior escribe:

- qué patrón es visible;
- qué NO puedes concluir;
- qué otra variable utilizarías para profundizar.

# 8. Comportamientos atípicos

Un valor atípico no necesariamente es un error.

Puede representar:

- error de captura;
- comportamiento excepcional;
- segmento muy pequeño;
- evento real poco frecuente.

Detectarlo es el primer paso. **Eliminarlo es otra decisión.**

## 8.1 Percentiles

In [ ]:
df["adr"].quantile([0.01, 0.25, 0.50, 0.75, 0.95, 0.99])

## 8.2 Método IQR

\[
IQR = Q_3 - Q_1
\]

Una regla común considera potencialmente atípicos los valores fuera de:

\[
[Q_1 - 1.5(IQR),\; Q_3 + 1.5(IQR)]
\]

In [ ]:
q1 = df["adr"].quantile(0.25)
q3 = df["adr"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1:", round(q1,2))
print("Q3:", round(q3,2))
print("IQR:", round(iqr,2))
print("Límite inferior:", round(lower,2))
print("Límite superior:", round(upper,2))

In [ ]:
adr_outliers = df[
    (df["adr"] < lower) |
    (df["adr"] > upper)
]

print("Observaciones potencialmente atípicas:", len(adr_outliers))
adr_outliers[["hotel", "adr", "lead_time", "total_nights"]].head(10)

### Pregunta

> ¿Debemos eliminar automáticamente todas estas observaciones?

La respuesta correcta normalmente es **no**.

Primero debemos investigar qué representan.

## Mini ejercicio 6

Detecta valores potencialmente atípicos en una de estas variables:

- `lead_time`
- `total_nights`
- `total_guests`

Describe qué encontraste y qué harías con ellos.

In [ ]:
# Tu código aquí

# 9. De segmentos manuales a similitud

Hasta ahora nosotros decidimos las reglas.

Ahora queremos que los datos ayuden a formar grupos.

Para hacerlo necesitamos una idea básica:

> observaciones parecidas deberían estar cerca entre sí.

## 9.1 Representar una reserva como punto

Por ejemplo:

\[
R = (lead\_time,\ adr)
\]

Una reserva podría ser:

\[
A=(20,100)
\]

y otra:

\[
B=(30,120)
\]

## 9.2 Distancia euclidiana

\[
d(A,B)=\sqrt{(x_A-x_B)^2+(y_A-y_B)^2}
\]

In [ ]:
A = np.array([20, 100])
B = np.array([30, 120])

distance = np.sqrt(
    np.sum((A - B) ** 2)
)

print("Distancia A-B:", round(distance,2))

Una distancia menor implica mayor similitud **respecto a las variables utilizadas**.

La similitud depende de qué características elegimos.

# 10. El problema de las escalas

In [ ]:
cluster_vars = [
    "lead_time",
    "adr",
    "total_nights",
    "total_guests"
]

df[cluster_vars].describe().T

Observa las escalas:

- `lead_time` puede tomar valores muy grandes;
- `total_guests` suele tomar valores pequeños.

Si calculamos distancias directamente, una variable podría dominar.

## 10.1 Estandarización

Usaremos:

$
z = \frac{x-\mu}{\sigma}
$

Después de estandarizar, las variables quedan en una escala comparable.

In [ ]:
X = (
    df[cluster_vars]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_df = pd.DataFrame(
    X_scaled,
    columns=cluster_vars
)

scaled_df.describe().round(2)

### Pregunta

> ¿Por qué K-Means podría producir segmentos engañosos si usamos variables con escalas muy diferentes?

# 11. K-Means — segmentación automática

K-Means será la técnica más avanzada de esta sesión.

La pregunta cambia de:

> “¿Cómo quiero dividir los datos?”

a:

> “¿Qué grupos parecen existir según la similitud entre observaciones?”

## 11.1 Idea del algoritmo

1. Elegimos un número de grupos \(k\).
2. Se inicializan \(k\) centroides.
3. Cada observación se asigna al centroide más cercano.
4. Los centroides se recalculan.
5. Se repite el proceso hasta estabilizarse.

El centroide representa el centro promedio de un cluster.

## 11.2 Visualización conceptual con dos variables

In [ ]:
simple_sample = (
    df[["lead_time", "adr"]]
    .dropna()
    .sample(min(1500, len(df)), random_state=42)
)

plt.figure(figsize=(9,6))
plt.scatter(
    simple_sample["lead_time"],
    simple_sample["adr"],
    alpha=0.25
)

plt.xlabel("Lead time")
plt.ylabel("ADR")
plt.ylim(0,500)
plt.title("¿Dónde parecen existir grupos?")
plt.show()

### Antes de ejecutar K-Means

Pregunta:

> Si tuvieras que dividir visualmente estos puntos en 2 o 3 grupos, ¿cómo lo harías?

## 11.3 Método del codo

In [ ]:
inertias = []
ks = range(2, 7)

for k in ks:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8,5))
plt.plot(
    list(ks),
    inertias,
    marker="o"
)

plt.xlabel("Número de clusters (k)")
plt.ylabel("Inercia")
plt.title("Método del codo")
plt.grid(alpha=0.25)
plt.show()

La inercia mide qué tan dispersas están las observaciones respecto a sus centroides.

Al aumentar \(k\), la inercia siempre disminuye.

Buscamos un punto donde agregar más clusters deje de aportar una mejora importante.

## 11.4 Comparar k = 2, 3 y 4

In [ ]:
profiles = {}

for k in [2, 3, 4]:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = km.fit_predict(X_scaled)

    temp = X.copy()
    temp["cluster"] = labels

    profiles[k] = (
        temp.groupby("cluster")[cluster_vars]
            .mean()
            .round(2)
    )

In [ ]:
print("k = 2")
display(profiles[2])

print("\nk = 3")
display(profiles[3])

print("\nk = 4")
display(profiles[4])

### Discusión

No debemos elegir \(k\) únicamente por una gráfica.

También buscamos:

- grupos suficientemente diferentes;
- perfiles fáciles de explicar;
- segmentos útiles para el análisis.

## 11.5 Trabajar con k = 3

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_scaled)

df.loc[X.index, "cluster_kmeans"] = cluster_labels

## 11.6 Perfil de los clusters

In [ ]:
cluster_profile = (
    df.dropna(subset=["cluster_kmeans"])
      .groupby("cluster_kmeans")
      .agg(
          reservas=("cluster_kmeans", "size"),
          lead_time_promedio=("lead_time", "mean"),
          adr_promedio=("adr", "mean"),
          noches_promedio=("total_nights", "mean"),
          huespedes_promedio=("total_guests", "mean"),
          cancelacion_pct=("is_canceled", lambda s: s.mean() * 100)
      )
      .round(2)
)

cluster_profile

## 11.7 Comparar perfiles visualmente

Como las variables originales tienen unidades diferentes, también resulta útil comparar los centroides en escala estandarizada.

In [ ]:
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=cluster_vars
)

plt.figure(figsize=(8,4))
sns.heatmap(
    centroids,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Perfil relativo de los clusters")
plt.xlabel("Variable")
plt.ylabel("Cluster")
plt.show()

### Cómo leer esta gráfica

Ahí sí se lee rápido:
- rojo/positivo → por encima del promedio;
- azul/negativo → por debajo;
- cerca de 0 → parecido al promedio.

## 11.8 Visualización de los clusters

In [ ]:
sample_clusters = (
    df.dropna(subset=["cluster_kmeans"])
      .sample(
          min(5000, df["cluster_kmeans"].notna().sum()),
          random_state=42
      )
)

plt.figure(figsize=(9,6))
sns.scatterplot(
    data=sample_clusters,
    x="lead_time",
    y="adr",
    hue="cluster_kmeans",
    palette="tab10",
    alpha=0.35
)

plt.title("Clusters K-Means: lead_time vs ADR")
plt.show()

In [ ]:
# Calcular el clúster más frecuente (más probable) para cada combinación de huéspedes y noches
pivot_cluster = sample_clusters.groupby(["total_guests", "total_nights"])["cluster_kmeans"].agg(lambda x: x.value_counts().index[0]).unstack(fill_value=np.nan)
pivot_counts = sample_clusters.groupby(["total_guests", "total_nights"]).size().unstack(fill_value=0)

# Filtrar rangos comunes para evitar una matriz demasiado dispersa y vacía
guest_range = range(1, 5)
night_range = range(1, 11)

pivot_cluster_filtered = pivot_cluster.reindex(index=guest_range, columns=night_range)
pivot_counts_filtered = pivot_counts.reindex(index=guest_range, columns=night_range, fill_value=0)

plt.figure(figsize=(12, 6))
# Usamos una paleta discreta con exactamente 3 colores para los clústeres 0, 1 y 2
sns.heatmap(
    pivot_cluster_filtered,
    annot=pivot_counts_filtered,
    fmt="d",
    cmap=sns.color_palette("tab10", 3),
    cbar_kws={
        'label': 'ID del Clúster Predominante',
        'ticks': [0, 1, 2]
    },
    vmin=-0.5, vmax=2.5
)

plt.title("Mapa de calor: Clúster predominante por Huéspedes vs Noches (Números indican cantidad de reservas)")
plt.xlabel("Total de noches")
plt.ylabel("Total de huéspedes")
plt.gca().invert_yaxis()  # Invertir la escala del eje Y (total de huéspedes)
plt.show()

## 11.9 Dar significado a los clusters

El algoritmo produce números:

- cluster 0;
- cluster 1;
- cluster 2.

Nuestro trabajo como analistas es convertirlos en **información interpretable**.

Ejemplo:

- reservas de alta anticipación;
- reservas cortas de tarifa alta;
- estancias más largas con más huéspedes.

Los nombres deben justificarse con el perfil real obtenido.

## Mini ejercicio 7

Para cada cluster:

1. describe sus características;
2. asigna un nombre;
3. identifica qué variables lo distinguen;
4. compara su tasa de cancelación;
5. explica para qué podría servir conocer ese segmento.

In [ ]:
# Escribe aquí tu interpretación

# 12. Limitaciones de K-Means

K-Means es útil, pero no es mágico.

Debemos recordar que:

- requiere elegir \(k\);
- depende de las variables seleccionadas;
- es sensible a la escala;
- los valores atípicos pueden afectar los centroides;
- favorece grupos relativamente compactos;
- un cluster matemático no siempre representa un segmento útil.

> **La interpretación sigue siendo responsabilidad del analista.**

# 13. De lo básico a K-Means

| Técnica | Pregunta |
|---|---|
| Frecuencias | ¿Qué aparece más? |
| Perfiles | ¿Cómo se comportan los grupos? |
| Tablas cruzadas | ¿Cómo se combinan dos categorías? |
| Categorías | ¿Cómo simplificamos una variable numérica? |
| Segmentación por reglas | ¿Qué perfiles podemos definir nosotros? |
| Visualización | ¿Qué patrones podemos observar? |
| Atípicos | ¿Qué observaciones son poco comunes? |
| Distancia | ¿Qué tan parecidas son dos observaciones? |
| Estandarización | ¿Cómo hacemos comparables las variables? |
| K-Means | ¿Qué segmentos emergen automáticamente? |

# 14. Actividad integradora final
## Tiempo sugerido: 60 minutos

### Reto

> Utiliza el dataset `hotel_bookings_clean.csv` para descubrir y explicar patrones relevantes en el comportamiento de las reservas.

No se trata de ejecutar todos los comandos vistos.

Se trata de construir un **análisis coherente**.

## Parte 1 — Formular una pregunta

Escribe una pregunta de explotación de información.

Ejemplos:

- ¿Qué tipos de reservas presentan comportamientos distintos?
- ¿Cómo cambia la cancelación entre diferentes perfiles?
- ¿Existen segmentos claros de reservas según anticipación, tarifa, duración y huéspedes?
- ¿Qué características distinguen a las reservas poco comunes?

### Tu pregunta

In [ ]:
# Escribe aquí tu pregunta como comentario

## Parte 2 — Identificar un patrón descriptivo

Utiliza frecuencias, proporciones, `groupby()` o `crosstab()`.

Debes incluir:

- una tabla;
- una visualización;
- una interpretación.

In [ ]:
# Tu análisis aquí

## Parte 3 — Crear una segmentación manual

Crea al menos **tres segmentos** utilizando reglas definidas por ti.

Después compara los segmentos mediante al menos dos métricas.

In [ ]:
# Tu segmentación aquí

## Parte 4 — Visualizar patrones

Incluye **al menos tres visualizaciones diferentes**.

Por ejemplo:

- barras;
- histograma;
- boxplot;
- scatter plot;
- heatmap.

Cada gráfica debe responder una pregunta.

In [ ]:
# Tus visualizaciones aquí

## Parte 5 — Identificar un comportamiento atípico

Elige una variable numérica y utiliza percentiles, IQR o boxplot.

Responde:

- ¿qué observaciones parecen atípicas?
- ¿qué porcentaje representan?
- ¿las eliminarías? ¿por qué?

In [ ]:
# Tu análisis de atípicos aquí

## Parte 6 — Aplicar K-Means

Utiliza como base:

- `lead_time`
- `adr`
- `total_nights`
- `total_guests`

Debes:

1. estandarizar;
2. utilizar el método del codo;
3. elegir un valor de \(k\);
4. generar clusters;
5. construir el perfil de cada cluster;
6. asignar un nombre descriptivo a cada uno.

In [ ]:
# Tu análisis K-Means aquí

## Parte 7 — Tres hallazgos finales

Redacta **tres hallazgos** utilizando esta estructura.

### Hallazgo 1
**Pregunta:**  
**Método:**  
**Evidencia:**  
**Interpretación:**  
**Utilidad:**  
**Limitación:**  

### Hallazgo 2
**Pregunta:**  
**Método:**  
**Evidencia:**  
**Interpretación:**  
**Utilidad:**  
**Limitación:**  

### Hallazgo 3
**Pregunta:**  
**Método:**  
**Evidencia:**  
**Interpretación:**  
**Utilidad:**  
**Limitación:**

# 15. Cierre

En esta sesión avanzamos desde herramientas muy sencillas hasta una técnica de segmentación automática.

La progresión fue:

> **Contar → comparar → categorizar → segmentar → visualizar → detectar atípicos → medir similitud → estandarizar → agrupar**

La idea central es:

> **Un algoritmo no es el resultado del análisis. El resultado es la información que podemos interpretar y comunicar a partir de los datos.**